# 1. Introdução

## 1.1. Contexto

Este notebook apresenta uma análise exploratória de um conjunto de dados sobre saúde mental de adolescentes e o uso de redes sociais. O objetivo é entender como variáveis como tempo de uso diário, sono, plataformas usadas e níveis de estresse, ansiedade e vício se relacionam com o rótulo de depressão.

## 1.2. Objetivos

- Carregar e inspecionar o dataset
- Limpar e preparar os dados para análise
- Explorar estatísticas descritivas de variáveis numéricas e categóricas
- Visualizar distribuições e padrões entre variáveis
- Identificar correlações e fatores associados ao rótulo de depressão
- Resumir as principais descobertas e limitações

# 2. Instalação e Importação das bibliotecas

In [ ]:
%pip install pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 3. Carregamento dos Dados

In [ ]:
# Carregar o dataset
path = 'data/Teen_Mental_Health_Dataset.csv'
df = pd.read_csv(path)

# Visão geral inicial
print('Dimensões do dataset:', df.shape)
print('\nAmostra inicial:')
display(df.head(10))

print('\nTipos de dados e valores não nulos:')
df.info()

print('\nDistribuição de depressão:')
print(df['depression_label'].value_counts(dropna=False))

print('\nValores únicos por coluna categórica:')
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    print(f'\n{col}:', df[col].value_counts().to_dict())

# 4. Limpeza e Preparação

In [ ]:
# Limpeza inicial e preparação dos dados

# Converter tipos numéricos e padronizar colunas categóricas
numeric_cols = ['age', 'daily_social_media_hours', 'sleep_hours', 'screen_time_before_sleep',
                'academic_performance', 'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Normalizar strings
string_cols = ['gender', 'platform_usage', 'social_interaction_level']
for col in string_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Padronizar gêneros
gender_map = {
    'male': 'male', 'm': 'male', 'man': 'male', 'masculino': 'male',
    'female': 'female', 'f': 'female', 'woman': 'female', 'feminino': 'female',
    'non-binary': 'other', 'nonbinary': 'other', 'other': 'other', 'trans': 'other',
    'prefer not to say': 'other', 'genderqueer': 'other', 'gender fluid': 'other',
}
df['gender'] = df['gender'].map(gender_map).fillna('other')

# Padronizar plataformas
platform_map = {
    'both': 'Both', 'instagram': 'Instagram', 'tiktok': 'TikTok', 'twitter': 'Twitter',
    'snapchat': 'Snapchat', 'facebook': 'Facebook', 'youtube': 'YouTube', 'whatsapp': 'WhatsApp'
}
df['platform_usage'] = df['platform_usage'].map(platform_map).fillna(df['platform_usage'].str.title())

# Padronizar nível de interação social
social_map = {'low': 'Low', 'medium': 'Medium', 'high': 'High'}
df['social_interaction_level'] = df['social_interaction_level'].map(social_map).fillna(df['social_interaction_level'].str.title())

# Converter rótulo em inteiro
if df['depression_label'].dtype != int:
    df['depression_label'] = pd.to_numeric(df['depression_label'], errors='coerce').astype('Int64')

# Identificar duplicatas e valores ausentes
duplicates = df.duplicated().sum()
missing_summary = df.isna().sum()

print('Duplicatas encontradas:', duplicates)
print('Valores ausentes por coluna:')
print(missing_summary)

# Não há tratamento agressivo aqui para manter o dataset íntegro;
# podemos continuar a análise mesmo com poucos valores ausentes.
df.drop_duplicates(inplace=True)
print('Dimensões após remoção de duplicatas:', df.shape)

# 5. Análise Exploratória

## 5.1. Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas gerais
print('Descrição numérica das variáveis:')
display(df[numeric_cols].describe().T)

print('\nContagem por rótulo de depressão:')
display(df['depression_label'].value_counts().rename('count').to_frame())

def categorical_summary(column):
    summary = df[column].value_counts().rename('count').to_frame()
    summary['percent'] = (summary['count'] / len(df)) * 100
    return summary

print('\nResumo das categorias de gênero:')
display(categorical_summary('gender'))

print('\nResumo de uso de plataforma:')
display(categorical_summary('platform_usage'))

print('\nResumo de interação social:')
display(categorical_summary('social_interaction_level'))

## 5.2. Distribuições

In [ ]:
# Distribuições das variáveis numéricas e categóricas

# Distribuições numéricas
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='#4472c4')
    ax.set_title(f'Distribuição de {col.replace("_", " ").title()}')
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

# Contagem de categorias relevantes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=df, x='gender', order=df['gender'].value_counts().index, ax=axes[0], palette='pastel')
axes[0].set_title('Distribuição de gênero')

sns.countplot(data=df, x='platform_usage', order=df['platform_usage'].value_counts().index, ax=axes[1], palette='pastel')
axes[1].set_title('Plataformas mais usadas')
axes[1].tick_params(axis='x', rotation=45)

sns.countplot(data=df, x='social_interaction_level', order=['Low', 'Medium', 'High'], ax=axes[2], palette='pastel')
axes[2].set_title('Nível de interação social')
plt.tight_layout()
plt.show()

# Boxplots por depressão
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plot_cols = ['daily_social_media_hours', 'sleep_hours', 'stress_level', 'anxiety_level']
for ax, col in zip(axes.flatten(), plot_cols):
    sns.boxplot(data=df, x='depression_label', y=col, ax=ax, palette='Set2')
    ax.set_title(f'{col.replace("_", " ").title()} por rótulo de depressão')
    ax.set_xlabel('Depressão (0 = não, 1 = sim)')
plt.tight_layout()
plt.show()

## 5.3. Correlações

In [ ]:
# Correlação entre as variáveis numéricas
corr_matrix = df[numeric_cols + ['depression_label']].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True, cbar_kws={'shrink': 0.8})
plt.title('Matriz de correlação entre variáveis numéricas')
plt.show()

# Correlação direta com o rótulo de depressão
target_corr = corr_matrix['depression_label'].drop('depression_label').sort_values(ascending=False)
print('Correlação das variáveis numéricas com o rótulo de depressão:')
print(target_corr)

# 6. Visualizações

In [ ]:
# Visualizações de relacionamentos importantes

# Distribuição do rótulo de depressão
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x='depression_label', palette=['#82c0ff', '#ff9999'])
ax.set_title('Distribuição do rótulo de depressão')
ax.set_xlabel('Depressão (0 = não, 1 = sim)')
ax.set_ylabel('Contagem')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=12, color='black', xytext=(0, 8), textcoords='offset points')
plt.show()

# Comparar médias por rótulo de depressão
compare_cols = ['daily_social_media_hours', 'sleep_hours', 'stress_level', 'anxiety_level', 'addiction_level']
mean_df = df.groupby('depression_label')[compare_cols].mean().reset_index()
mean_df_melted = mean_df.melt(id_vars='depression_label', var_name='feature', value_name='mean_value')
plt.figure(figsize=(12, 6))
sns.barplot(data=mean_df_melted, x='feature', y='mean_value', hue='depression_label', palette=['#82c0ff', '#ff9999'])
plt.title('Média das variáveis selecionadas por rótulo de depressão')
plt.xlabel('Variável')
plt.ylabel('Valor médio')
plt.xticks(rotation=45)
plt.legend(title='Depressão')
plt.tight_layout()
plt.show()

# Relação entre plataforma e depressão
plt.figure(figsize=(12, 6))
platform_rate = df.groupby('platform_usage')['depression_label'].mean().sort_values(ascending=False)
sns.barplot(x=platform_rate.index, y=platform_rate.values, palette='viridis')
plt.title('Taxa média de depressão por plataforma de uso')
plt.xlabel('Plataforma')
plt.ylabel('Proporção de depressão')
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# Relação entre interação social e depressão
plt.figure(figsize=(8, 5))
social_rate = df.groupby('social_interaction_level')['depression_label'].mean().reindex(['Low', 'Medium', 'High'])
sns.barplot(x=social_rate.index, y=social_rate.values, palette='magma')
plt.title('Taxa média de depressão por nível de interação social')
plt.xlabel('Nível de interação social')
plt.ylabel('Proporção de depressão')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 7. Conclusões

## Principais descobertas

- O dataset contém informações sobre comportamento de adolescentes em relação ao uso de redes sociais e seus indicadores de saúde mental.
- A variável `depression_label` é binária e permite observar diferenças claras em variáveis como horas de sono, níveis de estresse, ansiedade e dependência de redes sociais.
- A correlação mostra que `stress_level`, `anxiety_level` e `addiction_level` têm associação positiva mais forte com o rótulo de depressão.
- A plataforma de uso e a interação social também apresentam variações no percentual de depressão, sugerindo que hábitos e contexto social podem ser fatores importantes.

## Limitações

- O dataset é relativamente pequeno e não contém amostras de todas as faixas demográficas.
- Algumas colunas foram convertidas para categorias simplificadas, e a qualidade dos dados depende de respostas auto-relatadas.
- A análise aqui é exploratória e não estabelece relações causais.

## Próximos passos

- Construir um modelo de classificação para prever `depression_label` a partir das variáveis de comportamento.
- Aplicar técnicas de engenharia de features e validação cruzada para avaliar robustez.
- Coletar mais dados e adicionar informações demográficas adicionais para melhorar a generalização.